# Agriculture Data Cleaning: USDA-NASS Crop Chemical Application

Cleans the USDA-NASS Quick Stats **chemical & fertilizer application** export
(Iowa, state-level) into a tidy table: one row per
`(year, commodity, input_class, active_ingredient)`.

**Input:**  `data/tabular/01_raw/agriculture/Crop-Chemical-Application.csv`
**Output:** `data/tabular/02_clean/agriculture/crop-chemical-application-clean.csv`

**What's in it** — total pounds of active ingredient `APPLIED` to **corn** and
**soybeans** statewide (`SURVEY` program, selected years 2015-2023). The
substance is encoded in `Domain` / `Domain Category`:
- chemicals: `Domain = CHEMICAL, HERBICIDE` (or `FUNGICIDE`/`INSECTICIDE`/`OTHER`),
  `Domain Category = CHEMICAL, HERBICIDE: (GLYPHOSATE = 417300)` — an active
  ingredient name and its NASS chemical code, plus per-class `(TOTAL)` rows.
- fertilizer nutrients: `Domain = FERTILIZER`,
  `Domain Category = FERTILIZER: (NITROGEN)` — nutrient name, no code.

We pull the class, ingredient name, and numeric code into their own columns.
This is the most water-quality-relevant ag table: herbicide/insecticide and
N/P loads are direct candidate predictors of stream chemistry.

**Pipeline**
1. **Load** as strings. 2. **Drop** empty/constant columns. 3. **Parse**
`year` and `value` (suppression -> `NaN`). 4. **Unpack** `Data Item` (statistic
+ unit) and the `Domain` / `Domain Category` into
`input_class` / `active_ingredient` / `chemical_code`. 5. **Key**, check, save.

In [ ]:
import re
import numpy as np
import pandas as pd
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    """Walk upward until we find the repo's data/tabular directory.

    Notebooks have no __file__, and the kernel's working directory varies, so
    resolving paths relative to a fixed number of "../" is fragile. Searching
    upward for a sentinel makes the notebook runnable from anywhere.
    """
    here = (start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / "data" / "tabular").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate repo root containing data/tabular/")


REPO_ROOT = find_repo_root()
RAW_DIR = REPO_ROOT / "data" / "tabular" / "01_raw" / "agriculture"
CLEAN_DIR = REPO_ROOT / "data" / "tabular" / "02_clean" / "agriculture"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO_ROOT)
print("Raw dir:  ", RAW_DIR)
print("Clean dir:", CLEAN_DIR)

In [ ]:
# --- USDA-NASS shared cleaning helpers ------------------------------------
#
# NASS uses parenthetical letter codes in place of numbers. They are NOT data;
# they encode *why* a number is absent, so they must become NaN (never 0):
#   (D) withheld to avoid disclosing data for individual operations
#   (Z) value rounds to less than half the unit shown
#   (X) not applicable
#   (NA) not available
#   (H) sampling CV >= 99.95% (estimate too unreliable to publish)
#   (L) sampling CV  <  0.05%
# (D) appears in `Value`; (D)/(H)/(L) appear in `CV (%)`.
NASS_SUPPRESSION = {"(D)", "(Z)", "(X)", "(NA)", "(H)", "(L)", "(NA)", "(S)"}


def parse_nass_numeric(series: pd.Series) -> tuple[pd.Series, pd.Series]:
    """Parse a NASS Value/CV column into (float, was_suppressed_flag).

    Strips thousands separators, maps every suppression code to NaN, and flags
    which rows were a real suppression code (vs. genuinely blank) so downstream
    users can tell "censored" apart from "not collected".
    """
    s = series.astype("string").str.strip()
    suppressed = s.isin(NASS_SUPPRESSION)
    cleaned = s.mask(s.isin(NASS_SUPPRESSION))           # codes -> <NA>
    cleaned = cleaned.str.replace(",", "", regex=False)  # 1,234 -> 1234
    return pd.to_numeric(cleaned, errors="coerce"), suppressed.fillna(False)


# A Data Item is "<COMMODITY DETAIL> - <STATISTIC>[, MEASURED IN <UNIT>]",
# e.g. "CORN, GRAIN - YIELD, MEASURED IN BU / ACRE" or "CORN - ACRES PLANTED".
_DATA_ITEM_RE = re.compile(r"^(?P<detail>.+?) - (?P<stat>.+?)(?:, MEASURED IN (?P<unit>.+))?$")


def parse_data_item(item: str) -> tuple[str, str, str | None]:
    """Split a Data Item string into (commodity_detail, statistic, unit)."""
    m = _DATA_ITEM_RE.match(item)
    if not m:
        raise ValueError(f"Unparseable Data Item: {item!r}")
    return m.group("detail"), m.group("stat"), m.group("unit")

## Step 1 - Load

In [ ]:
RAW_FILE = "Crop-Chemical-Application.csv"
raw = pd.read_csv(RAW_DIR / RAW_FILE, dtype="string")
n_raw = len(raw)
print(f"Loaded {n_raw:,} rows x {raw.shape[1]} cols from {RAW_FILE}")
raw.head()

In [ ]:
# Every Data Item must parse, and every State ANSI must be Iowa (19) -- guard
# against a future re-pull silently changing the schema or scope.
assert raw["State ANSI"].dropna().eq("19").all(), "Non-Iowa rows present!"
_unparsed = [it for it in raw["Data Item"].dropna().unique() if not _DATA_ITEM_RE.match(it)]
assert not _unparsed, f"Unparseable Data Items: {_unparsed}"
print("Schema guards passed: all Iowa, all Data Items parse.")

## Step 2 - Drop structurally-empty and constant columns

This is a **state-level** table, so every county/ag-district/watershed column is
blank, along with the usual empties. `CV (%)` is entirely blank here too. We
keep `State ANSI` to stamp a `state_fips`.

In [ ]:
EMPTY_COLS = ["Week Ending", "Ag District", "Ag District Code", "County",
              "County ANSI", "Zip Code", "Region", "Watershed", "CV (%)"]
CONST_COLS = ["Program", "Period", "Geo Level", "State", "watershed_code"]

for col in EMPTY_COLS:
    assert raw[col].isna().all(), f"Expected {col!r} empty but it has values"
for col in CONST_COLS:
    assert raw[col].nunique(dropna=True) <= 1, f"Expected {col!r} constant: {raw[col].unique()}"

df = raw.drop(columns=EMPTY_COLS + CONST_COLS)
print(f"Dropped {len(EMPTY_COLS)} empty + {len(CONST_COLS)} constant columns; "
      f"{df.shape[1]} columns remain")

## Step 3 - Parse `year`, `state_fips`, and `value`

No `CV (%)` column survives, and there are no county keys, so this is short:
integer year, a constant 2-digit `state_fips`, and NASS-aware `value`.

In [ ]:
df["year"] = df["Year"].astype(int)
df["state_fips"] = df["State ANSI"].str.zfill(2)
df["value"], df["value_suppressed"] = parse_nass_numeric(df["Value"])
print(f"value: {df['value'].notna().sum():,} numeric, "
      f"{df['value_suppressed'].sum():,} suppressed (D) "
      f"({df['value_suppressed'].mean():.1%})")

## Step 4 - Unpack `Data Item` (statistic + unit)

Here the `Data Item` is simply `CORN - APPLICATIONS, MEASURED IN LB`. The
commodity duplicates the `Commodity` column, so we keep only the statistic and
unit from the parse.

In [ ]:
parsed = df["Data Item"].map(parse_data_item)
df["statistic"] = parsed.map(lambda t: t[1])   # APPLICATIONS
df["unit"] = parsed.map(lambda t: t[2])        # LB
print(df.groupby(["statistic", "unit"]).size().to_string())

## Step 5 - Unpack the substance from `Domain` / `Domain Category`

- `input_class` comes from `Domain`: the part after the comma for chemicals
  (`HERBICIDE`, `FUNGICIDE`, `INSECTICIDE`, `OTHER`) or `FERTILIZER`.
- `active_ingredient` and `chemical_code` come from the parenthetical in
  `Domain Category`: `(GLYPHOSATE = 417300)` -> name `GLYPHOSATE`, code `417300`.
  Per-class subtotals are `(TOTAL)` -> name `TOTAL`, no code. Fertilizer
  nutrients like `(NITROGEN)` have a name but no code.

In [ ]:
# input_class: drop a leading "CHEMICAL, " prefix if present, else use Domain as-is.
df["input_class"] = df["Domain"].str.replace(r"^CHEMICAL,\s*", "", regex=True)

inside = df["Domain Category"].str.extract(r":\s*\((?P<body>.+)\)$")["body"]
parts = inside.str.split("=", n=1, expand=True)
df["active_ingredient"] = parts[0].str.strip()
df["chemical_code"] = parts[1].str.strip() if parts.shape[1] > 1 else pd.NA

df = df.rename(columns={"Commodity": "commodity"})
print("input_class x active_ingredient coverage:")
print(df.groupby("input_class").agg(
    ingredients=("active_ingredient", "nunique"),
    with_code=("chemical_code", lambda s: s.notna().sum()),
    rows=("value", "size"),
).to_string())

## Step 6 - Select, order, and enforce a unique key

In [ ]:
OUTPUT_COLS = [
    "year", "state_fips", "commodity", "statistic", "unit",
    "input_class", "active_ingredient", "chemical_code",
    "value", "value_suppressed",
]
clean = df[OUTPUT_COLS].sort_values(
    ["year", "commodity", "input_class", "active_ingredient"]
).reset_index(drop=True)

KEY = ["year", "commodity", "input_class", "active_ingredient"]
dupes = clean.duplicated(KEY).sum()
assert dupes == 0, f"{dupes} duplicate key rows!"
print(f"Key is unique across {len(clean):,} rows.")
clean.head()

## Step 7 - Sanity check

In [ ]:
print(f"Rows: {len(clean):,}  |  years: {sorted(clean['year'].unique())}")
print(f"commodities: {clean['commodity'].unique().tolist()}")
print(f"suppressed: {clean['value_suppressed'].sum():,} "
      f"({clean['value_suppressed'].mean():.1%})\n")
print("Total lb applied by class (non-suppressed, excluding per-class TOTAL rows):")
detail = clean[(clean.active_ingredient != "TOTAL") & clean.value.notna()]
print(detail.groupby(["commodity", "input_class"])["value"]
            .agg(["count", "sum"]).to_string())

## Step 8 - Save

In [ ]:
out_file = CLEAN_DIR / "crop-chemical-application-clean.csv"
clean.to_csv(out_file, index=False)
print(f"Saved {len(clean):,} rows -> {out_file}")